# 03 — Modeling, Evaluation of deep neural network

This is the Phase 3/4 core of the project: train multiple classical ML models on the
TEP fault diagnosis task, evaluate them properly, tune the best one, and save it.

Task: **multi-class classification using deep neural network** — predict `faultNumber` (0 = normal, 1–20 = fault
type) from the 52 process sensor readings.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
#  Splits the dataset into training and testing sets to evaluate model performance.

from sklearn.preprocessing import LabelEncoder, StandardScaler
# LabelEncoder: Converts categorical labels (e.g., "cat", "dog") into numeric values.
# StandardScaler: Normalizes/standardizes numerical features so they have mean=0 and variance=1.

from sklearn.linear_model import Perceptron
# A simple linear classifier (single-layer neural network). Good for binary classification tasks.

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
# Lets you build a neural network layer-by-layer in a linear fashion.

from tensorflow.keras.layers import Dense
# Fully connected layer (every neuron connects to every neuron in the next layer).

from tensorflow.keras.layers import Dropout
# Randomly drops some neurons during training to prevent overfitting.

from tensorflow.keras.utils import to_categorical


In [3]:
path = '/content/drive/MyDrive/Sensornet.ai/data/raw/'

ff = pd.read_csv(path + 'TEP_FaultFree_Training.csv')
fy = pd.read_csv(path + 'TEP_Faulty_Training.csv')
train = pd.concat([ff, fy], ignore_index=True)
test  = pd.read_csv(path + 'TEP_Random_Testing_Sample_5_percent.csv')
print(train.shape, test.shape)

(5250000, 55) (504000, 55)


In [4]:
train

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,0,1,1,0.25038,3674.0,4529.0,9.2320,26.889,42.402,2704.3,...,53.744,24.657,62.544,22.137,39.935,42.323,47.757,47.510,41.258,18.447
1,0,1,2,0.25109,3659.4,4556.6,9.4264,26.721,42.576,2705.0,...,53.414,24.588,59.259,22.084,40.176,38.554,43.692,47.427,41.359,17.194
2,0,1,3,0.25038,3660.3,4477.8,9.4426,26.875,42.070,2706.2,...,54.357,24.666,61.275,22.380,40.244,38.990,46.699,47.468,41.199,20.530
3,0,1,4,0.24977,3661.3,4512.1,9.4776,26.758,42.063,2707.2,...,53.946,24.725,59.856,22.277,40.257,38.072,47.541,47.658,41.643,18.089
4,0,1,5,0.29405,3679.0,4497.0,9.3381,26.889,42.650,2705.1,...,53.658,28.797,60.717,21.947,39.144,41.955,47.645,47.346,41.507,18.461
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5249995,20,500,496,0.23419,3655.3,4461.7,9.3448,27.008,42.481,2703.0,...,53.670,23.350,61.061,20.719,40.999,38.653,47.386,47.528,40.212,17.659
5249996,20,500,497,0.26704,3647.4,4540.2,9.3546,27.034,42.671,2704.7,...,54.650,26.362,60.020,20.263,41.579,33.624,47.536,47.647,41.199,18.741
5249997,20,500,498,0.26543,3630.3,4571.6,9.4089,27.129,42.470,2705.1,...,54.274,26.521,59.824,20.189,41.505,40.967,52.437,47.802,41.302,23.199
5249998,20,500,499,0.27671,3655.7,4498.9,9.3781,27.353,42.281,2705.8,...,53.506,26.781,62.818,20.453,40.208,40.957,47.628,48.086,40.510,15.932


In [5]:
test

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,5,83,120,0.256870,3651.8,4462.2,9.3350,26.872,42.342,2710.5,...,54.563,25.775,61.981,22.148000,40.127,40.621,45.641,49.02900,41.102,19.221
1,13,106,935,0.211550,3624.5,4645.8,9.3730,26.794,42.202,2687.8,...,55.790,20.756,61.992,22.418000,39.475,42.738,42.716,29.09800,41.805,18.575
2,4,409,585,0.199850,3692.0,4545.3,9.2944,26.839,42.189,2716.7,...,54.354,19.875,62.156,23.129000,39.210,41.365,52.337,46.68100,45.674,19.235
3,15,460,858,0.274550,3648.3,4564.3,9.3293,26.951,42.317,2709.6,...,54.226,27.038,63.001,21.618000,40.121,40.008,43.444,43.19700,40.390,16.098
4,8,285,403,0.346420,3704.7,4598.6,8.7290,27.007,42.049,2723.8,...,54.774,33.868,57.092,23.264000,50.242,36.738,44.026,22.52500,39.813,19.108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503995,16,86,799,0.254020,3653.6,4462.3,9.3875,27.307,42.580,2708.4,...,53.419,24.769,59.657,22.562000,40.281,41.603,45.251,47.72300,40.930,18.940
503996,4,132,872,0.241700,3639.2,4555.5,9.4651,26.718,42.390,2707.3,...,53.375,24.111,59.673,22.474000,41.409,32.700,48.946,45.75500,44.769,17.819
503997,11,488,615,0.279210,3705.4,4538.8,9.3406,26.870,42.400,2702.0,...,53.922,27.621,59.917,21.414000,42.314,36.463,50.945,47.58000,46.074,17.279
503998,0,415,129,0.246620,3623.8,4497.8,9.4867,26.631,42.394,2698.3,...,53.519,24.230,61.846,22.454000,40.195,38.658,42.409,55.73400,40.755,21.022


In [6]:
train['faultNumber'].value_counts()

,count
faultNumber,
0,250000
1,250000
2,250000
3,250000
4,250000
5,250000
6,250000
7,250000
8,250000


In [4]:
x_train = train.drop(columns=['faultNumber','sample','simulationRun'])
y_train = train['faultNumber']
x_test  = test.drop(columns=['faultNumber','sample','simulationRun'])
y_test  = test['faultNumber']

In [5]:
import gc
for name in ['ff', 'fy', 'train', 'test']:
    if name in globals():
        del globals()[name]
gc.collect()

0

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled  = scaler.transform(x_test)

In [7]:
for name in ['x_train', 'x_test']:
    if name in globals():
        del globals()[name]
gc.collect()

0

In [8]:
X_train_scaled

array([[-0.0715966 ,  0.23897285,  0.20995561, ..., -0.17273188,
        -0.06653365, -0.07152586],
       [-0.06673719, -0.10232731,  0.46386585, ..., -0.17755922,
        -0.05619941, -0.31893978],
       [-0.0715966 , -0.08128826, -0.26106629, ..., -0.17517463,
        -0.07257048,  0.33977757],
       ...,
       [ 0.03140918, -0.78258995,  0.60186055, ..., -0.15574897,
        -0.06203161,  0.86679094],
       [ 0.10861218, -0.18882119, -0.06695375, ..., -0.13923135,
        -0.1430684 , -0.56813082],
       [ 0.09150158, -0.54648505, -0.29234508, ..., -0.13434585,
        -0.08116529, -0.60367312]])

In [9]:
X_test_scaled

array([[-0.0271775 , -0.27999041, -0.40458077, ..., -0.08438585,
        -0.08249544,  0.08130604],
       [-0.33735836, -0.91817495,  1.28447431, ..., -1.24358582,
        -0.01056506, -0.04625133],
       [-0.41743594,  0.65975386,  0.35990985, ..., -0.22094707,
         0.38530785,  0.08407044],
       ...,
       [ 0.12572277,  0.97300195,  0.30011215, ..., -0.16866064,
         0.42623553, -0.30215592],
       [-0.09733094, -0.93453865, -0.07707336, ...,  0.30558132,
        -0.1180002 ,  0.43692653],
       [-1.77092365,  3.41353184,  2.97260943, ..., -2.94746205,
         5.94594609,  5.38046599]])

In [10]:
#y_train_cat = to_categorical(y_train,num_classes = 21)
#y_test_cat = to_categorical(y_test,num_classes = 21)

In [11]:
#y_train_cat

In [12]:
model = keras.Sequential([
    layers.Input(shape=(52,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(21, activation='softmax')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=16, restore_best_weights=True)

In [13]:
for n, v in list(globals().items()):
    if isinstance(v, np.ndarray):
        print(n, round(v.nbytes / 1e9, 2), 'GB')
    elif isinstance(v, pd.DataFrame):
        print(n, round(v.memory_usage(deep=True).sum() / 1e9, 2), 'GB')

_ 0.21 GB
__ 2.18 GB
X_train_scaled 2.18 GB
X_test_scaled 0.21 GB
_8 2.18 GB
_9 0.21 GB


In [14]:
#X_train_scaled = X_train_scaled.astype('float32')
#X_test_scaled  = X_test_scaled.astype('float32')

In [14]:
history = model.fit(X_train_scaled,y_train,callbacks=[es],
                    epochs = 300,batch_size= 1024, validation_split = 0.2,verbose = 1)

Epoch 1/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.7288 - loss: 0.8350 - val_accuracy: 0.7758 - val_loss: 0.6089
Epoch 2/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7780 - loss: 0.6317 - val_accuracy: 0.7866 - val_loss: 0.5687
Epoch 3/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7822 - loss: 0.6100 - val_accuracy: 0.7853 - val_loss: 0.5648
Epoch 4/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7839 - loss: 0.6013 - val_accuracy: 0.7853 - val_loss: 0.5574
Epoch 5/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7852 - loss: 0.5958 - val_accuracy: 0.7896 - val_loss: 0.5525
Epoch 6/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7860 - loss: 0.5915 - val_accuracy: 0.7965 - val_loss: 0.5524
Epoch 7/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7871 - loss: 0.5875 - val_accuracy: 0.7922 - val_loss: 0.5491
Epoch 8/300
4102/4102 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.7880 -

In [15]:
nn_test_loss, nn_test_acc = model.evaluate(X_test_scaled, y_test, verbose=1)
print('NN test accuracy:', nn_test_acc)

15750/15750 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.7013 - loss: 1.2895
NN test accuracy: 0.701281726360321


In [ ]:
#rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
#rf.fit(X_train_scaled, y_train)
#print('RF test accuracy:', accuracy_score(y_test, rf.predict(X_test_scaled)))

RF test accuracy: 0.6579940476190477


In [16]:
import joblib
#joblib.dump(rf, '/content/drive/MyDrive/Sensornet.ai/models/rf_model.pkl')
model.save('/content/drive/MyDrive/Sensornet.ai/models/dense_model.keras')
joblib.dump(scaler, '/content/drive/MyDrive/Sensornet.ai/models/scaler.pkl')

['/content/drive/MyDrive/Sensornet.ai/models/scaler.pkl']

This project built a multi-class fault diagnosis pipeline on the Tennessee Eastman Process benchmark, classifying 21 operating conditions (normal operation plus 20 fault types) from 52 process variables. Identifier columns (simulationRun, sample) were excluded from the inputs to prevent leakage, and features were standardized using training-set statistics only.

A dense neural network (128 and 64 hidden units, trained with early stopping on the full training data) reached about 79% training and 80% validation accuracy, and 70.1% accuracy on the held-out test set (macro F1: [XX]). Gradient boosting (XGBoost) reached 80.8% accuracy and 0.817 macro F1 [on the grouped held-out split, confirm this], and the best tree-ensemble and linear baselines scored lower, so gradient boosting was the strongest model tested.

Test accuracy is lower than validation accuracy for the neural network. Test runs are longer and introduce faults much later than the training runs (roughly 8 hours versus 1 hour into the run), so a large share of each test run is normal operation carrying a fault label [verify by comparing accuracy before and after sample 160]. Faults 3, 9 and 15 are also widely reported as very hard to detect from the measured variables, which lowers recall for every model.
